In [1]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

In [2]:
from scipy.stats import norm, binom, poisson

In [6]:
p = poisson(mu=4)
print(f".cdf(3)    = {p.cdf(3):.4f}  ← P(3 or fewer events)")

.cdf(3)    = 0.4335  ← P(3 or fewer events)


In [7]:
print(1-0.4335)

0.5665


## Phase 2, Lesson 1 — The One-Sample t-Test

In [ ]:
import numpy as np
from scipy import stats

# Scenario: machine should dispense 250ml
# We measure 30 cups
# The command np.random.seed(42) ensures that the sequence of "random" numbers generated by the NumPy library 
# will be the same every time the code is run, which is crucial for reproducibility in experiments and debugging. 
np.random.seed(42)
sample = np.random.normal(loc=246, scale=10, size=30)  
# simulating a machine that actually runs low (mean=246)

# --- One-sample t-test ---
t_stat, p_value = stats.ttest_1samp(sample, popmean=250)

print(f"Sample mean:   {sample.mean():.3f}")
print(f"t-statistic:   {t_stat:.4f}")
print(f"p-value:       {p_value:.4f}")
print()

alpha = 0.05
if p_value <= alpha:
    print("Decision: REJECT H₀ — significant evidence the mean is not 250ml")
else:
    print("Decision: FAIL TO REJECT H₀ — no significant evidence against 250ml")

Sample mean:   244.119
t-statistic:   -3.5793
p-value:       0.0012

Decision: REJECT H₀ — significant evidence the mean is not 250ml


In [9]:
# --- What if the machine were actually fine? ---
np.random.seed(42)
good_sample = np.random.normal(loc=250, scale=10, size=30)

t2, p2 = stats.ttest_1samp(good_sample, popmean=250)
print(f"Good machine — p-value: {p2:.4f}")
print("Decision:", "REJECT H₀" if p2 <= 0.05 else "FAIL TO REJECT H₀")

Good machine — p-value: 0.2616
Decision: FAIL TO REJECT H₀


## Phase 2, Lesson 2 — Two-Sample t-Tests

In [10]:
import numpy as np
from scipy import stats

np.random.seed(42)

# -----------------------------------------------
# INDEPENDENT: two different groups
# Scenario: does a new teaching method improve scores?
# Group A: traditional method, Group B: new method
# -----------------------------------------------
group_a = np.random.normal(loc=70, scale=10, size=35)
group_b = np.random.normal(loc=75, scale=10, size=35)

t_ind, p_ind = stats.ttest_ind(group_a, group_b)

print("INDEPENDENT t-test")
print(f"  Group A mean: {group_a.mean():.2f}")
print(f"  Group B mean: {group_b.mean():.2f}")
print(f"  t-statistic:  {t_ind:.4f}")
print(f"  p-value:      {p_ind:.4f}")
print(f"  Decision:     {'REJECT H₀' if p_ind <= 0.05 else 'FAIL TO REJECT H₀'}")

print()

# -----------------------------------------------
# PAIRED: same people before and after
# Scenario: does a training programme improve performance?
# -----------------------------------------------
np.random.seed(42)
before = np.random.normal(loc=65, scale=8, size=30)
after  = before + np.random.normal(loc=5, scale=4, size=30)
# after = before + a real improvement of ~5 points + noise

t_rel, p_rel = stats.ttest_rel(before, after)

print("PAIRED t-test")
print(f"  Before mean:  {before.mean():.2f}")
print(f"  After mean:   {after.mean():.2f}")
print(f"  t-statistic:  {t_rel:.4f}")
print(f"  p-value:      {p_rel:.4f}")
print(f"  Decision:     {'REJECT H₀' if p_rel <= 0.05 else 'FAIL TO REJECT H₀'}")

INDEPENDENT t-test
  Group A mean: 68.67
  Group B mean: 73.63
  t-statistic:  -2.2891
  p-value:      0.0252
  Decision:     REJECT H₀

PAIRED t-test
  Before mean:  63.49
  After mean:   68.01
  t-statistic:  -6.6404
  p-value:      0.0000
  Decision:     REJECT H₀


## Phase 2, Lesson 3 — Non-Parametric Tests

In [11]:
import numpy as np
from scipy import stats

np.random.seed(42)

# Scenario: comparing recovery times (days) between two treatments
# Note: skewed data with an outlier — bad conditions for t-test
treatment_a = np.array([12, 14, 11, 15, 13, 16, 12, 14, 11, 95])  # outlier: 95
treatment_b = np.array([18, 20, 17, 22, 19, 21, 18, 20, 19, 21])

print("Group means:")
print(f"  Treatment A mean: {treatment_a.mean():.1f}  (distorted by outlier)")
print(f"  Treatment B mean: {treatment_b.mean():.1f}")

print()

# --- Parametric: independent t-test ---
t_stat, p_t = stats.ttest_ind(treatment_a, treatment_b)
print(f"t-test     p-value: {p_t:.4f}  → {'REJECT H₀' if p_t<=0.05 else 'FAIL TO REJECT H₀'}")

# --- Non-parametric: Mann-Whitney U ---
u_stat, p_mw = stats.mannwhitneyu(treatment_a, treatment_b, alternative='two-sided')
print(f"Mann-Whitney p-value: {p_mw:.4f}  → {'REJECT H₀' if p_mw<=0.05 else 'FAIL TO REJECT H₀'}")

print()

# --- Now paired: Wilcoxon signed-rank test ---
# Scenario: pain scores before and after treatment, small sample
before = np.array([7, 8, 6, 9, 7, 8, 5, 9, 6, 8])
after  = np.array([5, 6, 5, 7, 4, 6, 4, 7, 5, 6])

t_rel, p_rel   = stats.ttest_rel(before, after)
w_stat, p_wil  = stats.wilcoxon(before, after)

print(f"Paired t-test p-value: {p_rel:.4f}  → {'REJECT H₀' if p_rel<=0.05 else 'FAIL TO REJECT H₀'}")
print(f"Wilcoxon      p-value: {p_wil:.4f}  → {'REJECT H₀' if p_wil<=0.05 else 'FAIL TO REJECT H₀'}")

Group means:
  Treatment A mean: 21.3  (distorted by outlier)
  Treatment B mean: 19.5

t-test     p-value: 0.8292  → FAIL TO REJECT H₀
Mann-Whitney p-value: 0.0028  → REJECT H₀

Paired t-test p-value: 0.0000  → REJECT H₀
Wilcoxon      p-value: 0.0020  → REJECT H₀


## Phase 2, Lesson 4 — Chi-Square Tests

In [12]:
import numpy as np
from scipy import stats

# A die rolled 600 times. Fair die = 100 per face.
observed  = np.array([95, 102, 110, 88, 97, 108])
expected  = np.array([100, 100, 100, 100, 100, 100])

chi2_stat, p_val = stats.chisquare(f_obs=observed, f_exp=expected)

print("GOODNESS-OF-FIT TEST")
print(f"  Chi2 statistic: {chi2_stat:.4f}")
print(f"  p-value:        {p_val:.4f}")
print(f"  Decision: {'REJECT H₀ — die is likely loaded' if p_val<=0.05 else 'FAIL TO REJECT H₀ — die appears fair'}")

GOODNESS-OF-FIT TEST
  Chi2 statistic: 3.4600
  p-value:        0.6294
  Decision: FAIL TO REJECT H₀ — die appears fair


In [20]:
import numpy as np
from scipy import stats

contingency = np.array([
    [30, 50, 20],   # Under 30
    [40, 30, 30],   # Over 30
])

chi2, p, dof, expected = stats.chi2_contingency(contingency)

print("TEST OF INDEPENDENCE")
print(f"  Chi2 statistic: {chi2:.4f}")
print(f"  Degrees of freedom: {dof}")
print(f"  p-value:        {p:.4f}")
print(f"  Expected frequencies:\n{np.round(expected, 1)}")
print(f"  Decision: {'REJECT H0 — drink preference depends on age' if p <= 0.05 else 'FAIL TO REJECT H0 — no significant relationship'}")

TEST OF INDEPENDENCE
  Chi2 statistic: 8.4286
  Degrees of freedom: 2
  p-value:        0.0148
  Expected frequencies:
[[35. 40. 25.]
 [35. 40. 25.]]
  Decision: REJECT H0 — drink preference depends on age


In [24]:
np.random.seed(7)
sugar = np.random.normal(loc=85, scale=15, size=40)

t_stat, p_value = stats.ttest_1samp(sugar, popmean=80)

print(f"Sample mean:   {sample.mean():.3f}")
print(f"t-statistic:   {t_stat:.4f}")
print(f"p-value:       {p_value:.4f}")

Sample mean:   244.119
t-statistic:   1.3632
p-value:       0.1806


In [31]:

import numpy as np
from scipy import stats

np.random.seed(99)
sugar = np.random.normal(loc=85, scale=15, size=40)
sugar_city2 = np.random.normal(loc=80, scale=15, size=40)


print("Group means:")
print(f"  sugar mean: {sugar.mean():.1f}")
print(f"  sugar_city2 mean: {sugar_city2.mean():.1f}")

print()

# --- Parametric: independent t-test ---
t_stat, p_t = stats.ttest_ind(sugar, sugar_city2)
print(f"t-test     p-value: {p_t:.4f}  → {'REJECT H₀' if p_t<=0.05 else 'FAIL TO REJECT H₀'}")

# --- Non-parametric: Mann-Whitney U ---
u_stat, p_mw = stats.mannwhitneyu(sugar, sugar_city2, alternative='two-sided')
print(f"Mann-Whitney p-value: {p_mw:.4f}  → {'REJECT H₀' if p_mw<=0.05 else 'FAIL TO REJECT H₀'}")

print()


Group means:
  sugar mean: 85.1
  sugar_city2 mean: 81.2

t-test     p-value: 0.2505  → FAIL TO REJECT H₀
Mann-Whitney p-value: 0.2127  → FAIL TO REJECT H₀



In [30]:
import numpy as np
from scipy import stats

contingency = np.array([
    [ 40, 80],   # Under 30
    [ 20, 160],   # Over 30
])

chi2, p, dof, expected = stats.chi2_contingency(contingency)

print("TEST OF INDEPENDENCE")
print(f"  Chi2 statistic: {chi2:.4f}")
print(f"  Degrees of freedom: {dof}")
print(f"  p-value:        {p:.4f}")
print(f"  Expected frequencies:\n{np.round(expected, 1)}")
print(f"  Decision: {'REJECT H0 — drink preference depends on age' if p <= 0.05 else 'FAIL TO REJECT H0 — no significant relationship'}")

TEST OF INDEPENDENCE
  Chi2 statistic: 20.8550
  Degrees of freedom: 1
  p-value:        0.0000
  Expected frequencies:
[[ 24.  96.]
 [ 36. 144.]]
  Decision: REJECT H0 — drink preference depends on age
